# Лабораторная работа №4


## Наивный баевский метод


Наивный баевкий классификатор - вероятностный классификатор на основе теоремы Баеса
$$P(A|B)=P(B|A)\frac{P(A)}{P(B)}$$
Наивным он называется потому что мы приняли, что все признаки у нас независимые
Применяется в классификации текстов, фильтрации спама, тональности текста и рекомендательных системах



In [13]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import BernoulliNB, MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Загрузка данных из папок

def load_20news_from_folders(root_dir: Path):
    texts = []
    labels = []
    
    class_dirs = sorted([p for p in root_dir.iterdir() if p.is_dir()])
    target_names = [p.name for p in class_dirs]

    for class_dir in class_dirs:
        for file_path in sorted(class_dir.rglob("*")):
            if file_path.is_file():
                # Для корпуса 20 newsgroups обычно подходит latin1
                # errors='ignore' на случай "битых" символов
                text = file_path.read_text(encoding="latin1", errors="ignore")
                texts.append(text)
                labels.append(class_dir.name)

    return texts, labels, target_names


# Путь к текущей директории ноутбука
BASE_DIR = Path.cwd()

# Если ноутбук лежит в lab4, а данные в lab4/20news-bydate/...
DATA_DIR = BASE_DIR / "20news-bydate"
TRAIN_DIR = DATA_DIR / "20news-bydate-train"
TEST_DIR = DATA_DIR / "20news-bydate-test"

if not TRAIN_DIR.exists() or not TEST_DIR.exists():
    raise FileNotFoundError(
        f"Не найдены папки:\n{TRAIN_DIR}\n{TEST_DIR}\n"
        "Проверь, что lab4.ipynb лежит в той же директории, что и папка 20news-bydate."
    )

X_train_texts, y_train_labels, target_names = load_20news_from_folders(TRAIN_DIR)
X_test_texts, y_test_labels, _ = load_20news_from_folders(TEST_DIR)

print("Число документов в train:", len(X_train_texts))
print("Число документов в test:", len(X_test_texts))
print("Число рубрик:", len(target_names))
print("Рубрики:")
print(target_names)

Число документов в train: 11314
Число документов в test: 7532
Число рубрик: 20
Рубрики:
['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


In [14]:

# 2. Кодирование меток в числа


label_to_id = {label: i for i, label in enumerate(target_names)}
id_to_label = {i: label for label, i in label_to_id.items()}

y_train = np.array([label_to_id[label] for label in y_train_labels])
y_test = np.array([label_to_id[label] for label in y_test_labels])


# 3. Разделение train -> train/validation
# alpha подбираем по validation


X_train_texts_sub, X_val_texts, y_train_sub, y_val = train_test_split(
    X_train_texts,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print("\nРазмер обучающей части:", len(X_train_texts_sub))
print("Размер валидационной части:", len(X_val_texts))



Размер обучающей части: 9051
Размер валидационной части: 2263


In [15]:

# 4. Векторизация текста
# Для BernoulliNB нужны бинарные признаки
# Для MultinomialNB нужны счетчики слов
# Можно немного ограничить словарь и убрать очень редкие/частые токены, чтобы обучение было стабильнее.


vectorizer_mult = CountVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=2,
    max_df=0.95
)

vectorizer_bern = CountVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=2,
    max_df=0.95,
    binary=True
)

X_train_sub_mult = vectorizer_mult.fit_transform(X_train_texts_sub)
X_val_mult = vectorizer_mult.transform(X_val_texts)
X_test_mult = vectorizer_mult.transform(X_test_texts)

X_train_sub_bern = vectorizer_bern.fit_transform(X_train_texts_sub)
X_val_bern = vectorizer_bern.transform(X_val_texts)
X_test_bern = vectorizer_bern.transform(X_test_texts)

print("\nРазмерность Multinomial train:", X_train_sub_mult.shape)
print("Размерность Bernoulli train:", X_train_sub_bern.shape)



Размерность Multinomial train: (9051, 49200)
Размерность Bernoulli train: (9051, 49200)


In [16]:
# 5. Подготовка априорных вероятностей

n_classes = len(target_names)

# Равные вероятности
equal_priors = np.ones(n_classes) / n_classes

# Вероятности по долям классов в обучающей части
class_counts = np.bincount(y_train_sub, minlength=n_classes)
empirical_priors = class_counts / class_counts.sum()

print("\nРавные priors:")
print(equal_priors)

print("\nPriors по обучающей выборке:")
print(empirical_priors)


Равные priors:
[0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05
 0.05 0.05 0.05 0.05 0.05 0.05]

Priors по обучающей выборке:
[0.04242625 0.05159651 0.05225942 0.05214893 0.05115457 0.0523699
 0.05170699 0.05248039 0.05281184 0.05281184 0.05303281 0.05259087
 0.05225942 0.05248039 0.0523699  0.05292233 0.04828196 0.04982875
 0.04110043 0.03336648]


In [17]:
# 6. Подбор alpha и модели

# Сетка alpha из интервала (0, 1)
alpha_grid = np.round(np.arange(0.01, 1.00, 0.01), 2)

results = []

for alpha in alpha_grid:
    # ---- Multinomial + equal priors
    model = MultinomialNB(alpha=alpha, class_prior=equal_priors)
    model.fit(X_train_sub_mult, y_train_sub)
    pred_val = model.predict(X_val_mult)
    acc_val = accuracy_score(y_val, pred_val)

    results.append({
        "model": "MultinomialNB",
        "priors": "equal",
        "alpha": alpha,
        "val_accuracy": acc_val
    })

    # ---- Multinomial + empirical priors
    model = MultinomialNB(alpha=alpha, class_prior=empirical_priors)
    model.fit(X_train_sub_mult, y_train_sub)
    pred_val = model.predict(X_val_mult)
    acc_val = accuracy_score(y_val, pred_val)

    results.append({
        "model": "MultinomialNB",
        "priors": "empirical",
        "alpha": alpha,
        "val_accuracy": acc_val
    })

    # ---- Bernoulli + equal priors
    model = BernoulliNB(alpha=alpha, class_prior=equal_priors)
    model.fit(X_train_sub_bern, y_train_sub)
    pred_val = model.predict(X_val_bern)
    acc_val = accuracy_score(y_val, pred_val)

    results.append({
        "model": "BernoulliNB",
        "priors": "equal",
        "alpha": alpha,
        "val_accuracy": acc_val
    })

    # ---- Bernoulli + empirical priors
    model = BernoulliNB(alpha=alpha, class_prior=empirical_priors)
    model.fit(X_train_sub_bern, y_train_sub)
    pred_val = model.predict(X_val_bern)
    acc_val = accuracy_score(y_val, pred_val)

    results.append({
        "model": "BernoulliNB",
        "priors": "empirical",
        "alpha": alpha,
        "val_accuracy": acc_val
    })

results_df = pd.DataFrame(results).sort_values(
    by="val_accuracy", ascending=False
).reset_index(drop=True)

print("\nТоп-10 комбинаций по accuracy на validation:")
print(results_df.head(10))


Топ-10 комбинаций по accuracy на validation:
           model     priors  alpha  val_accuracy
0  MultinomialNB  empirical   0.04      0.886434
1  MultinomialNB      equal   0.04      0.886434
2  MultinomialNB  empirical   0.01      0.885108
3  MultinomialNB      equal   0.01      0.884666
4  MultinomialNB  empirical   0.03      0.884666
5  MultinomialNB      equal   0.06      0.884666
6  MultinomialNB  empirical   0.05      0.884666
7  MultinomialNB  empirical   0.06      0.884666
8  MultinomialNB  empirical   0.02      0.884666
9  MultinomialNB      equal   0.03      0.884224


In [18]:
# 7. Лучшая комбинация

best_row = results_df.iloc[0]
best_model_name = best_row["model"]
best_priors_name = best_row["priors"]
best_alpha = float(best_row["alpha"])

print("\nЛучшая модель:")
print(best_row)

if best_priors_name == "equal":
    best_priors = equal_priors
else:
    best_priors = empirical_priors


# 8. Обучение лучшей модели на ВСЕЙ train-выборке

if best_model_name == "MultinomialNB":
    final_vectorizer = CountVectorizer(
        lowercase=True,
        stop_words="english",
        min_df=2,
        max_df=0.95
    )
    X_train_final = final_vectorizer.fit_transform(X_train_texts)
    X_test_final = final_vectorizer.transform(X_test_texts)

    final_model = MultinomialNB(alpha=best_alpha, class_prior=best_priors)

else:
    final_vectorizer = CountVectorizer(
        lowercase=True,
        stop_words="english",
        min_df=2,
        max_df=0.95,
        binary=True
    )
    X_train_final = final_vectorizer.fit_transform(X_train_texts)
    X_test_final = final_vectorizer.transform(X_test_texts)

    final_model = BernoulliNB(alpha=best_alpha, class_prior=best_priors)

final_model.fit(X_train_final, y_train)
y_test_pred = final_model.predict(X_test_final)

test_acc = accuracy_score(y_test, y_test_pred)

print("\nИтоговая точность на test:", test_acc)
print("\nClassification report:\n")
print(classification_report(y_test, y_test_pred, target_names=target_names))


# 9. Сводная таблица по лучшей модели

summary_df = pd.DataFrame([{
    "best_model": best_model_name,
    "best_priors": best_priors_name,
    "best_alpha": best_alpha,
    "test_accuracy": test_acc
}])

print("\nИтог:")
print(summary_df)





Лучшая модель:
model           MultinomialNB
priors              empirical
alpha                    0.04
val_accuracy         0.886434
Name: 0, dtype: object

Итоговая точность на test: 0.8044344131704726

Classification report:

                          precision    recall  f1-score   support

             alt.atheism       0.81      0.83      0.82       319
           comp.graphics       0.57      0.80      0.67       389
 comp.os.ms-windows.misc       0.77      0.04      0.08       394
comp.sys.ibm.pc.hardware       0.53      0.76      0.63       392
   comp.sys.mac.hardware       0.73      0.84      0.78       385
          comp.windows.x       0.80      0.74      0.77       395
            misc.forsale       0.75      0.82      0.78       390
               rec.autos       0.87      0.89      0.88       396
         rec.motorcycles       0.92      0.96      0.94       398
      rec.sport.baseball       0.95      0.92      0.94       397
        rec.sport.hockey       0.96      0

F1-score – это сбалансированная оценка, которая одновременно учитывает: precision recall

Формула:
$$
F1=2⋅\frac{Precision+Recall}{Precision⋅Recall}
$$	​

Смысл
Он нужен, чтобы оценить качество не только по одному показателю.
Потому что бывает так:
precision высокий, а recall низкий;
или наоборот.
Тогда F1-score показывает более общий баланс между точностью и полнотой.

По результатам подбора параметров лучшей оказалась модель MultinomialNB с априорными вероятностями классов, соответствующими обучающей выборке, и коэффициентом сглаживания alpha = 0.04. 

Итоговая точность на тестовой выборке составила 0.804434, что соответствует 80,44% верно классифицированных документов. Наилучшее качество классификации достигнуто для рубрик **спортивной тематики**, где значения f1-score достигают 0.94–0.96. 

Наиболее проблемной оказалась рубрика comp.os.ms-windows.misc, для которой полнота составила 0.04, что свидетельствует о значительных затруднениях модели при распознавании документов данного класса.

Мультиномиальный наивный байесовский классификатор отличается от бернуллиевского способом представления признаков. В мультиномиальной модели учитывается число вхождений каждого слова в документ, тогда как в модели Бернулли фиксируется только факт присутствия или отсутствия слова. Для задачи рубрикации новостных статей мультиномиальная модель оказалась более эффективной, поскольку частота появления тематических слов является важным признаком принадлежности текста к определённой рубрике. 

Поэтому MultinomialNB показал более высокую точность классификации по сравнению с BernoulliNB.